# MeguvizCAC v0.4.0 R3 最终重训（A100-40GB / Drive 双写防丢版）

对应定稿方案：`docs/v0.4.0-final-training-plan.md`（基于 R1/R2 三轮消融反馈）。

### 本轮 R3 四项定稿修正
1. **主语料人设标签清洗（§3.1）**：下载冻结的 `v04-merged-1m4.jsonl.gz` 后本地执行 `meguviz_cac.v04_persona_labelfix`，仅清洗 `train` 中 12L 蒸馏/集成来源的 `196` 条低/中分 `本鱼/本鲸` 噪声样本（`score=max(score, 72)`、`train_weight=0.5`、`reason_code=A`），`val/test/probe` 39,494 行字节级零改动。
2. **结构化真人残差进主损失（§3.2）**：`train_v04_dualloop.py` 对 `structured_human_hard` 中 `y_sc <= 0.35` 样本追加二档尾部惩罚 `0.10 * relu(p - 0.20)^2`，定向压低 H06/H08 残差至 `< 0.35`。
3. **gate-guard 扩充至 92 对（§3.3）**：随仓库 `meguviz_cac/data/v04-gateguard-pairs.jsonl` 同步，在 R2 68 对基础上追加 24 对（A03 省 token/账单/缓存 +8、H06 评分表带口语收尾 +8、H08 词典引证带主观态度 +8，权重 2.2，零泄漏）。
4. **QAT 4 轮 + batch 384 + 防丢产物（§3.4 & §4）**：`epochs 2/1/4`、`batch 384`（A100-40GB 约 100 分钟）；产物训练完毕即刻打包写入 `/content/drive/MyDrive/MeguvizCAC/r3_v040/` 并自动触发浏览器 `files.download()`。

> **训练流程纪律**：请先手动连接 **A100 GPU** 运行时，然后依次运行 **Cell 1 → Cell 2 → Cell 3**。训练开始后**禁止触碰 Colab 运行时选择器**；若浏览器断线，重新打开后执行 **Cell 4** 即可续看进度或补下载。


## 1. 挂载 Google Drive、检查 A100 运行时、拉取 R3 最新代码与依赖


In [ ]:
import os, time, torch
from pathlib import Path
from google.colab import drive

# 1) 挂载 Google Drive 并验证可写（防实例更换丢权重）
drive.mount("/content/drive", force_remount=False)
DRIVE_OUT = Path("/content/drive/MyDrive/MeguvizCAC/r3_v040")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
(DRIVE_OUT / "_session_check.txt").write_text(
    f"mounted_utc={time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}\n",
    encoding="utf-8",
)
print(f"Drive backup directory ready: {DRIVE_OUT}")

# 2) 核查 A100 GPU 与系统内存
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!free -h | head -n 2
if not torch.cuda.is_available():
    raise RuntimeError("未检测到 GPU！请先在右上角选择 A100 GPU 运行时并连接后再运行本单元格。")
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
gpu_name = torch.cuda.get_device_name(0)
cc = torch.cuda.get_device_capability(0)
print(f"Detected GPU: {gpu_name} | cc={cc[0]}.{cc[1]} | VRAM={vram_gb:.1f} GB | bf16={torch.cuda.is_bf16_supported()}")

# 3) 拉取 GitHub main 最新 R3 代码与 92 对 gate-guard
REPO_DIR = "/content/MeguvizCAC"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/jichuo1/MeguvizCAC.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} pull --ff-only
%cd /content/MeguvizCAC
!git log -3 --oneline

for req in (
    "meguviz_cac/v04_persona_labelfix.py",
    "meguviz_cac/data/v04-gateguard-pairs.jsonl",
    "round01/bili-round01-user-confirmed.jsonl",
):
    assert Path(req).is_file(), f"Missing required repo asset: {req}"

# 4) 安装推理与量化导出依赖
!pip -q install -U onnx onnx_ir ml_dtypes onnxruntime transformers scipy scikit-learn huggingface_hub
print("STEP1_READY")


## 2. 下载 `v0.4-data` 冻结资产、执行 R3 人设清洗、92 对校验与 `dry-run` 自检


In [ ]:
import hashlib, json, shutil, subprocess
from pathlib import Path

%cd /content/MeguvizCAC

DATA_DIR = Path("/content/data")
DRIVE_OUT = Path("/content/drive/MyDrive/MeguvizCAC/r3_v040")
for d in (DATA_DIR / "stage2", DATA_DIR / "v03_teachers" / "e192", DATA_DIR / "co-teacher-gate"):
    d.mkdir(parents=True, exist_ok=True)

REL = "https://github.com/jichuo1/MeguvizCAC/releases/download"

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def ensure_download(url: str, dst: Path, expected_prefix: str | None = None) -> None:
    if dst.is_file() and dst.stat().st_size > 0:
        if expected_prefix is None or sha256_file(dst).startswith(expected_prefix):
            print(f"[cache hit] {dst.name}")
            return
    print(f"[download] {dst.name} ...")
    subprocess.run(["curl", "-fsSL", url, "-o", str(dst)], check=True)
    if expected_prefix is not None:
        got = sha256_file(dst)
        assert got.startswith(expected_prefix), f"SHA mismatch for {dst.name}: {got} != {expected_prefix}*"

# 1) 基础蒸馏包 & v0.3 词表
ensure_download(f"{REL}/v0.2-data/stage2_distill_bundle.zip", DATA_DIR / "stage2_distill_bundle.zip")
!unzip -qo /content/data/stage2_distill_bundle.zip -d /content/data/stage2

ensure_download(f"{REL}/v0.2-qwen-raw/stage2_qwen_bundle.zip", DATA_DIR / "stage2_qwen_bundle.zip")
!unzip -qoj /content/data/stage2_qwen_bundle.zip -d /content/data/stage2

ensure_download(f"{REL}/v0.3.0/meguvizcac-v0.3-loop-2x4-e192-int8.zip", DATA_DIR / "v03_teachers" / "e192.zip")
!unzip -qo /content/data/v03_teachers/e192.zip -d /content/data/v03_teachers/e192

# 2) v0.4-data 冻结训练资产
V04_ASSETS = [
    ("v04-qwen38-twins.jsonl", DATA_DIR / "v04-qwen38-twins.jsonl", "7ff4227c"),
    ("v04-inkling-wk-pairs.jsonl", DATA_DIR / "v04-inkling-wk-pairs.jsonl", "0e877a7c"),
    ("v04_semantic_intent_cache.npz", DATA_DIR / "v04_semantic_intent_cache.npz", "9880e38b"),
    ("v04-merged-1m4.jsonl.gz", DATA_DIR / "v04-merged-1m4.jsonl.gz", "11b63db0"),
    ("v04-twins-clean-pairs.jsonl", DATA_DIR / "co-teacher-gate" / "v04-twins-clean-pairs.jsonl", "3cae814d"),
]
for name, dst, sha_pref in V04_ASSETS:
    ensure_download(f"{REL}/v0.4-data/{name}", dst, sha_pref)

# 3) 同步仓库内 92 对 gate-guard 并核对 SHA256
gg_src = Path("/content/MeguvizCAC/meguviz_cac/data/v04-gateguard-pairs.jsonl")
gg_dst = DATA_DIR / "co-teacher-gate" / "v04-gateguard-pairs.jsonl"
shutil.copy2(gg_src, gg_dst)
gg_lines = [l for l in gg_dst.read_text(encoding="utf-8").splitlines() if l.strip()]
gg_sha = sha256_file(gg_dst)
assert len(gg_lines) == 92, f"Expected 92 gate-guard pairs, got {len(gg_lines)}"
assert gg_sha.startswith("98489d6b"), f"Unexpected gate-guard SHA256: {gg_sha}"
print(f"[OK] gateguard verified: {len(gg_lines)} pairs, sha256={gg_sha[:16]}")

# 4) 执行 §3.1 R3 人设标签后处理清洗（跨平台核验非训练集字节 SHA 与改写行数）
r3_corpus = DATA_DIR / "v04-merged-1m4-r3.jsonl.gz"
r3_report = DATA_DIR / "r3_persona_labelfix_report.json"
EXPECTED_NON_TRAIN_SHA = "6bb3f5bb77d70312d27ef81126583e6d97f9346fdd48ab905bbc462161e87bb5"
need_fix = True
if r3_corpus.is_file() and r3_report.is_file():
    try:
        cached = json.loads(r3_report.read_text(encoding="utf-8"))
        if (
            cached.get("modified_rows") == 196
            and cached.get("non_train_sha256") == EXPECTED_NON_TRAIN_SHA
            and cached.get("output_sha256") == sha256_file(r3_corpus)
        ):
            print(f"[cache hit] {r3_corpus.name} sha256={cached['output_sha256'][:16]}")
            need_fix = False
    except Exception:
        need_fix = True
if need_fix:
    !python -m meguviz_cac.v04_persona_labelfix \
      --in /content/data/v04-merged-1m4.jsonl.gz \
      --out /content/data/v04-merged-1m4-r3.jsonl.gz \
      --report /content/data/r3_persona_labelfix_report.json

rep = json.loads(r3_report.read_text(encoding="utf-8"))
assert rep["input_sha256"].startswith("11b63db0"), f"Unexpected input SHA: {rep['input_sha256']}"
assert rep["non_train_byte_identical"] is True, "Non-train rows changed!"
assert rep["non_train_sha256"] == EXPECTED_NON_TRAIN_SHA, f"Non-train SHA mismatch: {rep['non_train_sha256']}"
assert rep["modified_rows"] == 196, f"Expected 196 modified rows, got {rep['modified_rows']}"
assert rep["total_rows"] == 1411637, f"Unexpected total_rows: {rep['total_rows']}"
print(f"[OK] R3 persona labelfix verified: modified={rep['modified_rows']}, out_sha={rep['output_sha256'][:16]}")

if DRIVE_OUT.is_dir():
    shutil.copy2(r3_report, DRIVE_OUT / r3_report.name)
    shutil.copy2(gg_dst, DRIVE_OUT / gg_dst.name)

# 5) 预取 bge-small-zh-v1.5 底座权重，避免训练阶段网络抖动
!python -c "from transformers import AutoModel, AutoTokenizer; AutoTokenizer.from_pretrained('BAAI/bge-small-zh-v1.5'); AutoModel.from_pretrained('BAAI/bge-small-zh-v1.5'); print('BACKBONE_PREFETCH_OK')"

# 6) 完整 dry-run 自检（数学不变量、零泄漏、0 [UNK]、2357 干净对比对）
!python -m meguviz_cac.train_v04_dualloop --dry-run \
  --data /content/data/v04-merged-1m4-r3.jsonl.gz \
  --qwen-data /content/data/stage2/stage2-fused-train.jsonl.gz \
  --pairs-data /content/data/stage2/stage2-literary-pairs.jsonl \
  --extra-pairs /content/data/stage2/qwen38_27b_twins.jsonl \
  --extra-pairs /content/data/v04-qwen38-twins.jsonl \
  --extra-pairs /content/data/v04-inkling-wk-pairs.jsonl \
  --extra-pairs /content/data/co-teacher-gate/v04-twins-clean-pairs.jsonl \
  --extra-pairs /content/data/co-teacher-gate/v04-gateguard-pairs.jsonl \
  --intent-cache /content/data/v04_semantic_intent_cache.npz \
  --vocab /content/data/v03_teachers/e192/vocab.txt
print("STEP2_READY")


## 3. 启动 R3 全量重训（A100-80GB 满血策略：自动峰值探针 `bs=768..1024` + `torch.compile` + `sqrt LR` + `qat=5` + Drive 双写 + 自动下载）

- 自动调用仓库内 `meguviz_cac/run_r3_a100_80g.sh`：先做 ~20 秒最坏长度 Stage-3 QAT+EMA 显存与 `--compile` 探针，选取贴近 92% 显存上限的最大安全 `batch_size`（A100-80GB 目标 `768–1024`，自动匹配 `sqrt` LR、等比例 `pair_bs=36–48`、8-worker 数据流与 `qat-epochs 5`），全程预计 **25–45 分钟**。
- 使用 `setsid nohup` 后台运行，训练完成后驱动脚本**立刻打包同步至 Google Drive**，前台监控随后自动触发浏览器 `files.download()`。
- **注意**：训练期间切勿点击切换运行时！


In [ ]:
import json, os, shutil, subprocess, time
from pathlib import Path

DRIVE_OUT = Path("/content/drive/MyDrive/MeguvizCAC/r3_v040")
assert DRIVE_OUT.is_dir(), "Google Drive 未挂载！请先运行 Cell 1。"

DRIVER_SH = Path("/content/MeguvizCAC/meguviz_cac/run_r3_a100_80g.sh")
for req in (
    "/content/data/v04-merged-1m4-r3.jsonl.gz",
    "/content/data/co-teacher-gate/v04-gateguard-pairs.jsonl",
    "/content/data/v04_semantic_intent_cache.npz",
    "/content/MeguvizCAC/meguviz_cac/train_v04_dualloop.py",
    str(DRIVER_SH),
):
    assert Path(req).is_file(), f"前置资产缺失，请先运行 Cell 1 & Cell 2: {req}"

os.chmod(DRIVER_SH, 0o755)
subprocess.run(["bash", "-n", str(DRIVER_SH)], check=True)

PID_FILE = Path("/content/r3.pid")
DRIVER_LOG = Path("/content/log_r3_driver.txt")
TRAIN_LOG = Path("/content/log_r3_full.txt")

def pid_alive(pid: int) -> bool:
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False

if PID_FILE.is_file() and pid_alive(int(PID_FILE.read_text().strip())):
    pid = int(PID_FILE.read_text().strip())
    print(f"R3 driver already running, re-attaching pid={pid}")
else:
    Path("/content/r3_done.json").unlink(missing_ok=True)
    proc = subprocess.Popen(
        f"setsid nohup bash {DRIVER_SH} > {DRIVER_LOG} 2>&1 < /dev/null & echo $!",
        shell=True,
        stdout=subprocess.PIPE,
        text=True,
    )
    pid = int(proc.communicate()[0].strip())
    PID_FILE.write_text(str(pid), encoding="utf-8")
    time.sleep(5)
    print(f"Launched R3 A100-80GB background driver pid={pid}")

tick = 0
while pid_alive(pid):
    tick += 1
    now = time.strftime("%H:%M:%S")
    drv_tail = ""
    if DRIVER_LOG.is_file():
        lines = DRIVER_LOG.read_text(encoding="utf-8", errors="replace").splitlines()
        drv_tail = " | ".join(lines[-2:]) if lines else ""
    stage_line = ""
    if TRAIN_LOG.is_file():
        for ln in reversed(TRAIN_LOG.read_text(encoding="utf-8", errors="replace").splitlines()):
            if ln.startswith(("[Stage", "[Config")):
                stage_line = ln[:170]
                break
    gpu_stat = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.free,utilization.gpu", "--format=csv,noheader"],
        text=True,
    ).strip()
    ram_stat = subprocess.check_output(
        "free -g | awk '/Mem:/{print \"RAM \" $3 \"G/\" $2 \"G\"}'", shell=True, text=True
    ).strip()
    print(f"=== {now} (tick {tick}) === {gpu_stat} | {ram_stat}", flush=True)
    if drv_tail:
        print(f"  driver: {drv_tail}", flush=True)
    if stage_line:
        print(f"  train : {stage_line}", flush=True)
    if tick % 5 == 0 and TRAIN_LOG.is_file():
        try:
            shutil.copy2(TRAIN_LOG, DRIVE_OUT / "log_r3_full_live.txt")
        except Exception as e:
            print(f"  [warn] live log sync: {e}", flush=True)
    time.sleep(90)

print("\n=== R3 DRIVER EXITED ===")
if DRIVER_LOG.is_file():
    print(DRIVER_LOG.read_text(encoding="utf-8", errors="replace"))

done_file = Path("/content/r3_done.json")
if done_file.is_file():
    info = json.loads(done_file.read_text(encoding="utf-8"))
    print("R3_DONE_SUMMARY:", json.dumps(info, ensure_ascii=False, indent=2))
    full_zip = Path(info["full_zip"])
    if full_zip.is_file():
        print(f"Triggering automatic browser download: {full_zip.name}")
        from google.colab import files
        files.download(str(full_zip))
else:
    raise RuntimeError("R3 训练未生成完成标记 r3_done.json，请检查上方 log_r3_driver.txt 与 /content/log_r3_full.txt！")


## 4. 断线重连监控 / 手动补下载单元（防失联保险）

如果训练中途浏览器断线，重新打开页面后直接运行本单元格即可查看后台训练进度，或直接从 `/content/` 与 Google Drive 校验并补下载产物 zip。


In [ ]:
import json, os, subprocess
from pathlib import Path

PID_FILE = Path("/content/r3.pid")
DRIVER_LOG = Path("/content/log_r3_driver.txt")
TRAIN_LOG = Path("/content/log_r3_full.txt")
DRIVE_OUT = Path("/content/drive/MyDrive/MeguvizCAC/r3_v040")

alive = False
if PID_FILE.is_file():
    pid = int(PID_FILE.read_text().strip())
    try:
        os.kill(pid, 0)
        alive = True
        print(f"R3 driver is STILL RUNNING (pid={pid})")
    except OSError:
        print(f"R3 driver pid={pid} has exited.")

if alive:
    !nvidia-smi --query-gpu=memory.used,memory.free,utilization.gpu --format=csv,noheader
    if DRIVER_LOG.is_file():
        print("--- driver tail ---")
        print("\n".join(DRIVER_LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-8:]))
    if TRAIN_LOG.is_file():
        print("--- train latest ---")
        print("\n".join(TRAIN_LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-15:]))
else:
    if DRIVER_LOG.is_file():
        print(DRIVER_LOG.read_text(encoding="utf-8", errors="replace"))
    zip_candidates = [
        Path("/content/meguvizcac-v0.4.0-r3-dualloop-2x4-e192-full.zip"),
        DRIVE_OUT / "meguvizcac-v0.4.0-r3-dualloop-2x4-e192-full.zip",
    ]
    found = next((p for p in zip_candidates if p.is_file()), None)
    if found is not None:
        print(f"Found artifact zip: {found} ({found.stat().st_size / (1024**2):.2f} MB)")
        if DRIVE_OUT.is_dir():
            print("Drive contents:", [p.name for p in sorted(DRIVE_OUT.iterdir())])
        from google.colab import files
        files.download(str(found))
    else:
        print("未找到产物 zip，请检查 /content/log_r3_full.txt 或确认 Drive 已挂载。")
